
### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [5]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: coffee_culture_rag_sample.pdf
  ✓ Loaded 1 pages

Processing: quantum_computing_rag.pdf
  ✓ Loaded 1 pages

Processing: sustainable_urban_planning_rag.pdf
  ✓ Loaded 1 pages

Total documents loaded: 3


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260312164027', 'source': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'coffee_culture_rag_sample.pdf', 'file_type': 'pdf'}, page_content="Introduction to Coffee Culture\n1. Origins of Coffee\nCoffee was discovered in the Ethiopian highlands. Legend has it that a goat herder named Kaldi noticed his\ngoats became energetic after eating berries from a certain tree. This led to the cultivation of coffee beans,\nwhich eventually spread to the Arabian Peninsula and then worldwide.\n2. Regional Varieties\nEthiopia: Known for Yirgacheffe beans, which offer floral and citrus notes.\nBrazil: The world's largest coffee producer, known for nutty and chocolatey flavor profiles.\nVietnam: Famous for its high-quality Robusta beans and the traditional Ca Phe Sua (coffee with condensed\nmilk).\n3. Popular Brewing Methods

In [7]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 3 documents into 5 chunks

Example chunk:
Content: Introduction to Coffee Culture
1. Origins of Coffee
Coffee was discovered in the Ethiopian highlands. Legend has it that a goat herder named Kaldi noticed his
goats became energetic after eating berri...
Metadata: {'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260312164027', 'source': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'coffee_culture_rag_sample.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260312164027', 'source': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'coffee_culture_rag_sample.pdf', 'file_type': 'pdf'}, page_content="Introduction to Coffee Culture\n1. Origins of Coffee\nCoffee was discovered in the Ethiopian highlands. Legend has it that a goat herder named Kaldi noticed his\ngoats became energetic after eating berries from a certain tree. This led to the cultivation of coffee beans,\nwhich eventually spread to the Arabian Peninsula and then worldwide.\n2. Regional Varieties\nEthiopia: Known for Yirgacheffe beans, which offer floral and citrus notes.\nBrazil: The world's largest coffee producer, known for nutty and chocolatey flavor profiles.\nVietnam: Famous for its high-quality Robusta beans and the traditional Ca Phe Sua (coffee with condensed\nmilk).\n3. Popular Brewing Methods

### Embedding and vector  store DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer #This is a library for generating sentence embeddings, which are dense vector representations of sentences. It is built on top of the Hugging Face Transformers library and provides an easy-to-use interface for generating embeddings for various natural language processing tasks.
import chromadb
from chromadb.config import Settings
import uuid #This is a library for generating universally unique ids
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
        """Initializes the embedding manager
        
        Args:
            model_name: Hugging Face model name for generating sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model() # protectded method to load the model
    
    def _load_model(self):
        """Loads the SentenceTransformer model"""
        try:
            print(f"⏳ Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"✅ Loaded embedding model: {self.model_name} and embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"❌ Error loading model: {e}")
            raise
    
    def generate_embeddings(self,documents:List[str]) -> np.ndarray:
        """Generates embeddings for a list of documents
        
        Args:
            texts: List of document texts to generate embeddings for.
        Returns:
            Numpy array of shape (num_documents, embedding_dimension) containing the generated embeddings.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embeddings.")
        try:
            print(f"⏳ Generating embeddings for {len(documents)} documents...")
            embeddings = self.model.encode(documents, show_progress_bar=True)
            print(f"✅ Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"❌ Error generating embeddings: {e}")
            raise
        
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

⏳ Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7722.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded embedding model: all-MiniLM-L6-v2 and embedding dimension: 384


### VectorStore

In [20]:
class VectorStore:
    """Manages document embeddings in a chromadb vector store"""
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the chromadb collection to store embeddings.
            persist_directory: Directory where the chromadb database will be persisted in the hard disk.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None 
        self._initialize_vector_store() # protected method to initialize the vector store
        
    def _initialize_vector_store(self):
        """Initializes the chromadb vector store and collection"""
        try:
            # create chromadb client with persistence settings
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory) # create a persistent client that will save the vector store to disk
            
            # Get or create a collection for storing document embeddings
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Pdf document embeddings for RAG system"}
            )
            print(f"✅ Initialized vector store with collection: {self.collection_name} and persistence directory: {self.persist_directory}")
            print(f"Current number of embeddings in the collection: {self.collection.count()}")
        except Exception as e:
            print(f"❌ Error initializing vector store: {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """Adds documents and their corresponding embeddings to the vector store
        
        Args:
            documents: List of document objects containing metadata and content.
            embeddings: Numpy array of shape (num_documents, embedding_dimension):
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must be the same.")
        
        print(f"⏳ Adding {len(documents)} documents and embeddings to the vector store...")
        
        ids = []
        metadatas = []
        documents_text = []
        embedding_lists = []
        
        for i, (doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}" # generate a unique id for each document 
            ids.append(doc_id)
            
            #Prepare metadata for the document
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Documne Content
            documents_text.append(doc.page_content)
            
            # Embding
            embedding_lists.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_lists,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"✅ Successfully added {len(documents)} documents to the vector store. Total embeddings in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"❌ Error adding documents to vector store: {e}")
            raise
        
## Initialize the vector store
vectorstore = VectorStore()
vectorstore

✅ Initialized vector store with collection: pdf_documents and persistence directory: ../data/vector_store
Current number of embeddings in the collection: 0


In [12]:
chunks

[Document(metadata={'producer': 'PyFPDF 1.7.2 http://pyfpdf.googlecode.com/', 'creator': 'PyPDF', 'creationdate': 'D:20260312164027', 'source': '..\\data\\pdf\\coffee_culture_rag_sample.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'coffee_culture_rag_sample.pdf', 'file_type': 'pdf'}, page_content="Introduction to Coffee Culture\n1. Origins of Coffee\nCoffee was discovered in the Ethiopian highlands. Legend has it that a goat herder named Kaldi noticed his\ngoats became energetic after eating berries from a certain tree. This led to the cultivation of coffee beans,\nwhich eventually spread to the Arabian Peninsula and then worldwide.\n2. Regional Varieties\nEthiopia: Known for Yirgacheffe beans, which offer floral and citrus notes.\nBrazil: The world's largest coffee producer, known for nutty and chocolatey flavor profiles.\nVietnam: Famous for its high-quality Robusta beans and the traditional Ca Phe Sua (coffee with condensed\nmilk).\n3. Popular Brewing Methods

In [21]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings) 

⏳ Generating embeddings for 5 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.53it/s]

✅ Generated embeddings with shape: (5, 384)
⏳ Adding 5 documents and embeddings to the vector store...
✅ Successfully added 5 documents to the vector store. Total embeddings in collection: 5


### Retriever Pipeline From VectorStore

In [22]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)



In [23]:
rag_retriever

In [24]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
⏳ Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.69it/s]

✅ Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]